<a href="https://www.kaggle.com/code/martinsertin/blueberry-yield-optuna-xgb?scriptVersionId=287479434" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import optuna
import warnings
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
warnings.filterwarnings("ignore")


In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])


In [3]:
def create_features(df):
    data = df.copy()
    
    data['total_bees'] = data['honeybee'] + data['bumbles'] + data['andrena'] + data['osmia']
    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee_inter'] = data['osmia'] * data['honeybee']
    
    
    data['temp_range'] = data['MaxOfUpperTRange'] - data['MinOfLowerTRange']
    data['avg_temp'] = (data['AverageOfUpperTRange'] + data['AverageOfLowerTRange']) / 2
    
    for col in ['clonesize', 'osmia', 'honeybee', 'RainingDays', 'temp_range']:
        data[f'{col}_sq'] = data[col] ** 2
    

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)
    

    cluster_cols = ['clonesize', 'total_bees', 'avg_temp', 'RainingDays']
    scaler = StandardScaler()
    scaled = scaler.fit_transform(data[cluster_cols])
    data['cluster'] = KMeans(n_clusters=8, random_state=42, n_init='auto').fit_predict(scaled)
    
    return data

train_fe = create_features(train)
test_fe = create_features(test)

y = train_fe['yield']
X = train_fe.drop('yield', axis=1)

min_y, max_y = y.min(), y.max()


In [4]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 600, 2500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': 42,
        'tree_method': 'hist',
    }
    
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    mae_scores = []
    
    for train_idx, val_idx in kf.split(X):
        model = XGBRegressor(**params)
        model.fit(X.iloc[train_idx], y.iloc[train_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                  early_stopping_rounds=100,
                  verbose=False)
        pred = model.predict(X.iloc[val_idx])
        mae_scores.append(mean_absolute_error(y.iloc[val_idx], pred))
    
    return np.mean(mae_scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)   

best_params = study.best_trial.params
print("Best params:", best_params)
print("Best CV MAE:", study.best_value)

[I 2025-12-20 15:36:35,308] A new study created in memory with name: no-name-10797339-1dec-488c-a40a-3b84815b241d
[I 2025-12-20 15:36:59,551] Trial 0 finished with value: 248.49298048667447 and parameters: {'n_estimators': 2104, 'learning_rate': 0.006791194068254415, 'max_depth': 5, 'min_child_weight': 2, 'subsample': 0.8251645452860072, 'colsample_bytree': 0.9537819538451372, 'gamma': 1.325084996647881, 'reg_alpha': 4.827627158326899, 'reg_lambda': 0.7322378778097238}. Best is trial 0 with value: 248.49298048667447.
[I 2025-12-20 15:37:21,845] Trial 1 finished with value: 250.28816939173961 and parameters: {'n_estimators': 2050, 'learning_rate': 0.00851042579737138, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.947317809685812, 'colsample_bytree': 0.6613632711113856, 'gamma': 0.30709169175155704, 'reg_alpha': 4.494408060396406, 'reg_lambda': 0.24234242395762218}. Best is trial 0 with value: 248.49298048667447.
[I 2025-12-20 15:37:30,830] Trial 2 finished with value: 250.268433

Best params: {'n_estimators': 991, 'learning_rate': 0.012241113938144723, 'max_depth': 5, 'min_child_weight': 10, 'subsample': 0.731218705264012, 'colsample_bytree': 0.8434298848825986, 'gamma': 0.735588819112134, 'reg_alpha': 0.15225920762193665, 'reg_lambda': 1.106374582832366}
Best CV MAE: 248.24718056113016


In [5]:
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, timeout=None)
best_params = study.best_trial.params
print(best_params)
print(f"The best Cv in MAE: {study.best_value:.4f}")

[I 2025-12-20 16:00:51,501] A new study created in memory with name: no-name-ff7f8175-10d5-49cb-90ff-d485d7590ec1
[I 2025-12-20 16:00:59,436] Trial 0 finished with value: 255.4969217378802 and parameters: {'n_estimators': 1312, 'learning_rate': 0.0862735828664018, 'max_depth': 9, 'min_child_weight': 8, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.005575058716044}. Best is trial 0 with value: 255.4969217378802.
[I 2025-12-20 16:01:53,395] Trial 1 finished with value: 253.18655249097395 and parameters: {'n_estimators': 1946, 'learning_rate': 0.005318033256270142, 'max_depth': 10, 'min_child_weight': 10, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 1.5212112147976886, 'reg_lambda': 2.6237821581611893}. Best is trial 1 with value: 253.18655249097395.
[I 2025-12-20 16:02:15,240] Trial 2 finished with value: 251.60083366

{'n_estimators': 1384, 'learning_rate': 0.01078514219712489, 'max_depth': 5, 'min_child_weight': 3, 'subsample': 0.7312202514997275, 'colsample_bytree': 0.9588891518978466, 'gamma': 4.952751004280436, 'reg_alpha': 3.568847250091892, 'reg_lambda': 2.591544085923073}
The best Cv in MAE: 248.1499


In [6]:
kf = KFold(n_splits=15, shuffle=True, random_state=42)
test_preds = np.zeros(len(test_fe))
oof_preds = np.zeros(len(X))
mae_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}/15")
    
    model = XGBRegressor(**best_params, random_state=42, tree_method='hist')
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              early_stopping_rounds=100, verbose=False)
    
    oof_preds[val_idx] = model.predict(X.iloc[val_idx])
    test_preds += model.predict(test_fe) / 15
    
    mae_scores.append(mean_absolute_error(y.iloc[val_idx], oof_preds[val_idx]))

print(f"Mean OOF MAE: {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")

Fold 1/15
Fold 2/15
Fold 3/15
Fold 4/15
Fold 5/15
Fold 6/15
Fold 7/15
Fold 8/15
Fold 9/15
Fold 10/15
Fold 11/15
Fold 12/15
Fold 13/15
Fold 14/15
Fold 15/15
Mean OOF MAE: 247.9126 ± 7.2166


In [7]:
test_preds = np.clip(test_preds, min_y, max_y)

submission = pd.DataFrame({'id': test_ids, 'yield': test_preds})
submission.to_csv('submission.csv', index=False)

submission.head()

,id,yield
0,15000,7452.128387
1,15001,5836.695374
2,15002,6748.123627
3,15003,4645.780579
4,15004,5870.734528
